# CS781 Capstone - Phase 1: Training-corpus overlap check (measurement only)

Question: do the official test prompts (and our held-out prompts) have verbatim passages in the training files?

- For every prompt, two anchors: its last 12 words, and the 12 words ending 50 words earlier.
- All four training files are streamed once; every occurrence of an anchor in a training document is found (Aho-Corasick).
- Held-out prompts: their own document is recognised by its key and ignored (and reported as a sanity check), so only matches in *other* documents count.
   - The 128 words after a match are scored with BERTScore against the true continuation: an upper bound on what retrieval could add.
- Test prompts: how many have such matches.

- No model, no generation, no submission.
- Attach the decoding (or pool) notebook output that contains `heldout_eval_set.json`.
- Internet on, `HF` secret.
- CPU works; a GPU only speeds up the final BERTScore step.
- Expected time ~30-60 min (mostly reading 18 GB of JSON).

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("HuggingFace login successful.")

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyahocorasick", "ijson", "bert_score"], check=True)
import os, json, time, glob, hashlib
import numpy as np
from collections import Counter, defaultdict
import ahocorasick, ijson, torch
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"
ANCHOR_WORDS = 12          # a match = these consecutive words of the prompt found verbatim in a training document
CONT_WORDS = 128
MAX_CANDS = 5              # retrieved continuations kept per prompt
OUTPUT_DIR = "/kaggle/working/overlap_check"
os.makedirs(OUTPUT_DIR, exist_ok=True)
T0 = time.time()

# ---- prompts: the 1000 official test prompts + the held-out prompts (with their own document key) ----
with open(hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE, repo_type="dataset", token=HF_TOKEN), encoding="utf-8") as f:
    test_data = json.load(f)
_held = sorted(glob.glob("/kaggle/input/**/heldout_eval_set.json", recursive=True))
assert _held, "attach the decoding (or pool) notebook output that contains heldout_eval_set.json"
heldout = json.load(open(_held[-1]))
P = [{"kind": "test", "id": str(k), "prompt": v["text"], "doc_key": None, "reference": None} for k, v in test_data.items()]
P += [{"kind": "heldout", "id": f"h{i}", "prompt": e["prompt"], "doc_key": e["doc_key"], "reference": e["reference"]}
      for i, e in enumerate(heldout)]
print(f"prompts: {sum(p['kind'] == 'test' for p in P)} test, {sum(p['kind'] == 'heldout' for p in P)} held-out ({_held[-1]})")

# Two anchors per prompt: its last ANCHOR_WORDS words ("end": the text after a match is a candidate continuation) and the
# ANCHOR_WORDS words ending 50 words earlier ("mid": catches documents whose text diverges only near the prompt end).
A = ahocorasick.Automaton()
anchors = defaultdict(list)
for pi, p in enumerate(P):
    w = p["prompt"].split()
    if len(w) >= ANCHOR_WORDS:
        anchors[" " + " ".join(w[-ANCHOR_WORDS:]) + " "].append((pi, "end", 0))
    if len(w) >= ANCHOR_WORDS + 50:
        anchors[" " + " ".join(w[-ANCHOR_WORDS - 50:-50]) + " "].append((pi, "mid", 50))
for key, owners in anchors.items():
    A.add_word(key, owners)
A.make_automaton()
print(f"anchors: {len(anchors):,}")

# ---- stream every training document once ----
found = [dict(end=0, mid=0, self_found=False, cands=[]) for _ in P]
stats = Counter()
for fi, train_file in enumerate(TRAIN_FILES):
    src = train_file.split("/")[-1].replace("train_", "").replace(".json", "")
    path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=train_file, repo_type="dataset", token=HF_TOKEN)
    t0 = time.time()
    with open(path, "rb") as fh:
        for di, rec in enumerate(ijson.items(fh, "item")):
            text = rec["text"]
            norm = " " + " ".join(text.split()) + " "
            hits = list(A.iter(norm))
            stats[f"docs_{src}"] += 1
            if not hits:
                continue
            key = hashlib.md5(text[:500].encode()).hexdigest()          # same document key as Arch 2's pipeline
            for end_idx, owners in hits:
                for pi, kind, skip in owners:
                    if P[pi]["doc_key"] == key:                           # a held-out prompt's own document: not retrieval
                        found[pi]["self_found"] = True
                        continue
                    found[pi][kind] += 1
                    if len(found[pi]["cands"]) < MAX_CANDS:
                        after = norm[end_idx + 1:end_idx + 1 + 12 * (skip + CONT_WORDS)].split()
                        cont = after[skip:skip + CONT_WORDS]
                        if len(cont) == CONT_WORDS:
                            found[pi]["cands"].append({"source": src, "doc": di, "anchor": kind, "text": " ".join(cont)})
            if di % 50000 == 0 and di:
                print(f"  {src}: {di:,} docs | {(time.time() - t0)/60:.1f} min | total {(time.time() - T0)/60:.1f} min")
    print(f"done {src}: {stats[f'docs_{src}']:,} docs in {(time.time() - t0)/60:.1f} min")

# ---- report ----
def frac(xs):
    return f"{sum(xs)}/{len(xs)} ({100 * np.mean(xs):.1f}%)" if xs else "n/a"
for kind in ("test", "heldout"):
    idx = [i for i, p in enumerate(P) if p["kind"] == kind]
    print(f"\n{kind.upper()} prompts: {len(idx)}")
    print(f"  last {ANCHOR_WORDS} words found in another training document:      {frac([found[i]['end'] > 0 for i in idx])}")
    print(f"  words ending 50 earlier found in another training document: {frac([found[i]['mid'] > 0 for i in idx])}")
    print(f"  with at least one full {CONT_WORDS}-word retrieved continuation:    {frac([len(found[i]['cands']) > 0 for i in idx])}")
    if kind == "heldout":
        print(f"  (sanity) prompt found in its own document: {frac([found[i]['self_found'] for i in idx])}")
    by_src = Counter(c["source"] for i in idx for c in found[i]["cands"][:1])
    print(f"  source of the first retrieved continuation: {dict(by_src)}")

# Held-out upper bound: BERTScore of the retrieved continuations against the true continuation
hi = [i for i, p in enumerate(P) if p["kind"] == "heldout" and found[i]["cands"]]
if hi:
    from bert_score import BERTScorer
    sc = BERTScorer(model_type="roberta-large", lang="en", device="cuda" if torch.cuda.is_available() else "cpu")
    pairs = [(i, c["text"]) for i in hi for c in found[i]["cands"]]
    _, _, f1 = sc.score([t for _, t in pairs], [P[i]["reference"] for i, _ in pairs])
    best, first = defaultdict(float), {}
    for (i, _), v in zip(pairs, f1.tolist()):
        best[i] = max(best[i], v)
        first.setdefault(i, v)
    n_all = sum(p["kind"] == "heldout" for p in P)
    print(f"\nHELD-OUT retrieval quality on the {len(hi)}/{n_all} prompts with a retrieved continuation:")
    print(f"  BERTScore F1 of the first retrieved continuation: mean {np.mean(list(first.values())):.4f}")
    print(f"  BERTScore F1 of the best of up to {MAX_CANDS}:              mean {np.mean(list(best.values())):.4f}")
    print(f"  share with first-retrieved F1 > 0.95 (near-identical text): {frac([v > 0.95 for v in first.values()])}")
    print(f"  (Arch 2 + MBR on these held-out prompts: about 0.862 on average)")
json.dump({"prompts": [{k: p[k] for k in ("kind", "id", "doc_key")} for p in P], "found": found, "stats": dict(stats)},
          open(os.path.join(OUTPUT_DIR, "overlap_matches.json"), "w"))
print(f"\nSaved {OUTPUT_DIR}/overlap_matches.json | total {(time.time() - T0)/60:.1f} min")
